<a href="https://colab.research.google.com/github/zachdaube/CS485/blob/main/Homework_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **CS485 & CS584 - Homework 3**


In this Colab, we will work to construct our own graph neural network (GNN) using PyTorch Geometric (PyG) and then apply that model on two [Open Graph Benchmark (OGB)](https://ogb.stanford.edu/) datasets. These two datasets will be used to benchmark your model's performance on two different graph-based tasks: 1) node property prediction, predicting properties of single nodes and 2) graph property prediction, predicting properties of entire graphs or subgraphs.

First, we'll review how PyTorch Geometric stores graphs as tensors. Since OGB offers benchmark datasets and evaluation tools for graph learning, complementing PyG, we'll load and inspect an OGB dataset using the `ogb` package, which provides both data loaders and evaluators for large-scale, diverse graph benchmarks.

Then, we will build our own graph neural network using PyTorch Geometric. We will then train and evaluate our model on the OGB node property prediction and graph property prediction tasks.

Now let's get started! This Colab should take 1-2 hours to complete.

**Note**: Make sure to **restart and run all** before submission, so that the intermediate variables / packages will carry over to the next cell.

# Device
You might need to use a GPU for this Colab to run quickly.

Please click `Runtime` and then `Change runtime type`. Then set the `hardware accelerator` to **GPU**. You can switch to a `T4 GPU` instance to access a free T4 GPU with 16 GB of memory.

# Setup
Let's start by downloading the required packages.

In [ ]:
import torch
import os
print("PyTorch has version {}".format(torch.__version__))

PyTorch has version 2.5.1+cu124


Download the necessary packages for PyG. Make sure that your version of torch matches the output from the cell above. In case of any issues, more information can be found on the [PyG's installation page](https://pytorch-geometric.readthedocs.io/en/latest/notes/installation.html).

In [ ]:
torch_version = str(torch.__version__)
scatter_src = f"https://pytorch-geometric.com/whl/torch-{torch_version}.html"
sparse_src = f"https://pytorch-geometric.com/whl/torch-{torch_version}.html"
!pip install torch-scatter -f $scatter_src
!pip install torch-sparse -f $sparse_src
!pip install torch-geometric
!pip install ogb

Check if we can use GPU. If you use GPU, the device should be `cuda`.

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device: {}'.format(device))

Device: cuda


# 1 Load Dataset: PyTorch Geometric & Open Graph Benchmark (OGB)


Recall that PyTorch Geometric has two classes for storing and/or transforming graphs into tensor format. One is [`torch_geometric.datasets`](https://pytorch-geometric.readthedocs.io/en/latest/modules/datasets.html), which contains a variety of common graph datasets. Another is [`torch_geometric.data`](https://pytorch-geometric.readthedocs.io/en/latest/modules/data.html), which provides the data handling of graphs in PyTorch tensors.

Another package we'll use is **OGB**, which provides **standardized**, **large-scale**, and **challenging benchmark datasets** with consistent evaluation protocols. It offers a variety of datasets tailored for different graph tasks, including [Node Classification](https://ogb.stanford.edu/docs/nodeprop/), [Link Prediction](https://ogb.stanford.edu/docs/linkprop/), and [Graph Classification](https://ogb.stanford.edu/docs/graphprop/). Additionally, OGB features leaderboards for each task, such as the [Node Classification Leaderboard](https://ogb.stanford.edu/docs/leader_nodeprop/), allowing for easy comparison of model performance.

Before diving into to graph deep learning, we will first learn how to use these datasets.

## PyG Datasets

The `torch_geometric.datasets` class has many common graph datasets. Here we will explore its usage through one example dataset.

In [ ]:
from torch_geometric.datasets import TUDataset

root = './enzymes'
name = 'ENZYMES'

# The ENZYMES dataset
pyg_dataset= TUDataset(root, name)
# You will find that there are 600 graphs in this dataset,
# which means this pyg_dataset is a dataset for graph classification
print(pyg_dataset)

# Let us check the number of classes and number of features in the ENZYMES dataset
num_classes = pyg_dataset.num_classes
num_features = pyg_dataset.num_features
print("{} dataset has {} classes".format(name, num_classes))
print("{} dataset has {} features".format(name, num_features))

Processing...


ENZYMES(600)
ENZYMES dataset has 6 classes
ENZYMES dataset has 3 features


Done!


## PyG Data

Now, we know that each PyG dataset stores a list of torch_geometric.data.Data objects, where each torch_geometric.data.Data object represents a graph. We can easily get the Data object by indexing into the dataset. Now, let's examine the details of a graph from a PyG dataset.

For more information such as what is stored in the `Data` object, please refer to the [documentation](https://pytorch-geometric.readthedocs.io/en/latest/modules/data.html#torch_geometric.data.Data).

In [ ]:
from torch_geometric.datasets import TUDataset

root = './enzymes'
name = 'ENZYMES'
pyg_dataset= TUDataset(root, name)

print(pyg_dataset[0])
# Check the label of the graph with index 100 in the ENZYMES dataset
label = pyg_dataset[100].y.item()
# Check the number of edges the graph with index 200 have
num_edges = pyg_dataset[0].num_edges

print(f'Graph with index 100 has label {label}.')
print(f'Graph with index 100 has {num_edges} edges.')

Data(edge_index=[2, 168], x=[37, 3], y=[1])
Graph with index 100 has label 4
Graph with index 100 has 168 edges


## OGB Datasets

The Open Graph Benchmark (OGB) is a collection of realistic, large-scale, and diverse benchmark datasets for machine learning on graphs. Its datasets are automatically downloaded, processed, and split using the OGB Data Loader. The model performance can then be evaluated by using the OGB Evaluator in a unified manner.

OGB also supports PyG dataset and data classes. Here we take a look on the `ogbn-arxiv` dataset.

In [ ]:
import torch_geometric.transforms as T
from ogb.nodeproppred import PygNodePropPredDataset

dataset_name = 'ogbn-arxiv'
# Load the dataset and transform it to sparse tensor
dataset = PygNodePropPredDataset(name=dataset_name, transform=T.ToSparseTensor())
# You will find that there is only 1 graph in this dataset,
# which means this dataset is for node classification
print(f'The {dataset_name} dataset has {len(dataset)} graph')

# Extract the graph
data = dataset[0]
# Check the number of features
num_features = data.num_features
print(data)
print(f'The graph has {num_features} features')

The ogbn-arxiv dataset has 1 graph
Data(num_nodes=169343, x=[169343, 128], node_year=[169343, 1], y=[169343, 1], adj_t=[169343, 169343, nnz=1166243])
The graph has 128 features


# 2 GNN: Node Property Prediction

In this section we will build our first graph neural network using PyTorch Geometric. Then we will apply it to the task of node property prediction (node classification).

**Node Property Prediction** is a fundamental task in graph machine learning, where the goal is to predict specific properties or labels associated with the nodes in a graph. Formally, given a graph $G = (V, E)$, where $V$ is the set of nodes and $E$ is the set of edges, the objective is to learn a function $f: V \rightarrow \mathcal{Y}$ that maps each node $v \in V$ to a target label $y_v \in \mathcal{Y}$.

- Let $G = (V, E, X)$ be a graph where:  
  - $V = \{v_1, v_2, \dots, v_n\}$ is the set of **nodes**,  
  - $E \subseteq V \times V$ is the set of **edges**,  
  - $X \in \mathbb{R}^{n \times d}$ represents the **feature matrix**, where $x_i \in \mathbb{R}^d$ is the feature vector for node $v_i$.

- The task is to predict node labels $Y = \{y_1, y_2, \dots, y_n\}$, where $y_i \in \mathcal{Y}$ is the label or property associated with node $v_i$.

#### **Common Applications:**
- **Social Networks:** Predicting user attributes like age, interests, or political affiliation.  
- **Biological Networks:** Classifying proteins or genes in protein-protein interaction networks.  
- **Citation Networks:** Predicting research topics or categories of academic papers.


Specifically, we will use GCN as the foundation for your graph neural network ([Kipf et al. (2017)](https://arxiv.org/pdf/1609.02907.pdf)). To do so, we will work with PyG's built-in `GCNConv` layer.

## Setup

In [ ]:
import torch
import pandas as pd
import torch.nn.functional as F
import torch_geometric.transforms as T
print(torch.__version__)

2.5.1+cu124


## Load and Preprocess the Dataset

In [ ]:
from ogb.nodeproppred import PygNodePropPredDataset, Evaluator

dataset_name = 'ogbn-arxiv'
dataset = PygNodePropPredDataset(name=dataset_name, transform=T.ToSparseTensor())
data = dataset[0]
# Make the adjacency matrix to symmetric (undirected graph)
data.adj_t = data.adj_t.to_symmetric()
device = 'cuda' if torch.cuda.is_available() else 'cpu'

data = data.to(device)
split_idx = dataset.get_idx_split()
train_idx = split_idx['train'].to(device)

/usr/local/lib/python3.11/dist-packages/ogb/nodeproppred/dataset_pyg.py:69: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.data, self.slices = torch.load(self.processed_

## GCN Model

Please follow the figure below to implement the `forward` function.


![test](https://drive.google.com/uc?id=128AuYAXNXGg7PIhJJ7e420DoPWKb-RtL)

- **GCNConv:** The `GCNConv` layer performs **graph convolution**, aggregating feature information from neighboring nodes to capture the local structure of the graph and update node embeddings accordingly. For more information please refer to the [GCN documentation](https://pytorch-geometric.readthedocs.io/en/latest/generated/torch_geometric.nn.conv.GCNConv.html#torch_geometric.nn.conv.GCNConv). We have various other GNN layers, such as `GraphSAGEConv`, `GATConv`, and `GINConv`, as detailed in the [PyG GNN layers documentation](https://pytorch-geometric.readthedocs.io/en/latest/modules/nn.html#convolutional-layers).

- **Batch Normalization (BN):** **Batch Normalization** normalizes the inputs of each layer to have a consistent distribution, which **accelerates training** and improves **model stability** by reducing internal covariate shift. For more information please refer to the [BN documentation](https://pytorch.org/docs/stable/generated/torch.nn.BatchNorm1d.html).

- **ReLU:** The **ReLU (Rectified Linear Unit)** activation function introduces **non-linearity** into the model, allowing it to learn complex patterns and relationships in the graph data.

- **Dropout:** **Dropout** is a regularization technique that randomly **deactivates neurons** during training to prevent **overfitting**, enhancing the model's generalization ability on unseen data. For more information about ReLU and Dropout, please refer to the [`torch.nn.functional` documentation](https://pytorch.org/docs/stable/nn.functional.html ).

- **LogSoftmax (in Final Projection Layer):** The **LogSoftmax** function converts the final output scores into **log-probabilities**, making it suitable for **multi-class classification** tasks and ensuring numerical stability when combined with negative log-likelihood loss.

Now let us implement our GCN model!

In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
import torch_geometric.transforms as T


class GCN(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers,
                 dropout, return_embeds=False):
        # TODO: Implement a function that initializes self.convs,
        # self.bns, and self.softmax.

        super(GCN, self).__init__()

        # A list of GCNConv layers
        self.convs = None

        # A list of 1D batch normalization layers
        self.bns = None

        # The log softmax layer
        self.softmax = None

        ############# Your code here ############
        ## Note:
        ## 1. You should use torch.nn.ModuleList for self.convs and self.bns
        ## 2. self.convs has num_layers GCNConv layers
        ## 3. self.bns has num_layers - 1 BatchNorm1d layers
        ## 4. You should use torch.nn.LogSoftmax for self.softmax
        ## 5. The parameters you can set for GCNConv include 'in_channels' and 'out_channels'.
        ## 6. The only parameter you need to set for BatchNorm1d is 'num_features'
        ## (~10 lines of code)
        #########################################

        # Probability of an element getting zeroed
        self.dropout = dropout

        # Skip classification layer and return node embeddings
        self.return_embeds = return_embeds

    def reset_parameters(self):
        for conv in self.convs:
            conv.reset_parameters()
        for bn in self.bns:
            bn.reset_parameters()

    def forward(self, x, adj_t):
        # TODO: Implement a function that takes the feature tensor x and
        # edge_index tensor adj_t and returns the output tensor as
        # shown in the figure.

        out = None

        ############# Your code here ############
        ## Note:
        ## 1. Construct the network as shown in the figure
        ## 2. torch.nn.functional.relu and torch.nn.functional.dropout are useful
        ## 3. Don't forget to set F.dropout training to self.training
        ## 4. If return_embeds is True, then skip the last softmax layer
        ## (~7 lines of code)
        #########################################

        return out

In [ ]:
def train(model, data, train_idx, optimizer, loss_fn):
    # TODO: Implement a function that trains the model by
    # using the given optimizer and loss_fn.
    model.train()
    loss = 0

    ############# Your code here ############
    ## Note:
    ## 1. Zero grad the optimizer
    ## 2. Feed the data into the model
    ## 3. Slice the model output and label by train_idx
    ## 4. Feed the sliced output and label to loss_fn
    ## (~4 lines of code)
    #########################################

    loss.backward()
    optimizer.step()

    return loss.item()

In [ ]:
# Test function here
@torch.no_grad()
def test(model, data, split_idx, evaluator, save_model_results=False):
    # TODO: Implement a function that tests the model by
    # using the given split_idx and evaluator.
    model.eval()

    # The output of model on all data
    out = None

    ############# Your code here ############
    ## (~1 line of code)
    ## Note:
    ## 1. No index slicing here
    #########################################

    y_pred = out.argmax(dim=-1, keepdim=True)

    train_acc = evaluator.eval({
        'y_true': data.y[split_idx['train']],
        'y_pred': y_pred[split_idx['train']],
    })['acc']
    valid_acc = evaluator.eval({
        'y_true': data.y[split_idx['valid']],
        'y_pred': y_pred[split_idx['valid']],
    })['acc']
    test_acc = evaluator.eval({
        'y_true': data.y[split_idx['test']],
        'y_pred': y_pred[split_idx['test']],
    })['acc']

    if save_model_results:
      print ("Saving Model Predictions")

      data = {}
      data['y_pred'] = y_pred.view(-1).cpu().detach().numpy()

      df = pd.DataFrame(data=data)
      # Save locally as csv
      df.to_csv('ogbn-arxiv_node.csv', sep=',', index=False)

    return train_acc, valid_acc, test_acc

In [ ]:
# Please DO NOT change the args
args = {
    'device': device,
    'num_layers': 3,
    'hidden_dim': 256,
    'dropout': 0.5,
    'lr': 0.01,
    'epochs': 100,
}
args

In [ ]:
model = GCN(data.num_features, args['hidden_dim'],
            dataset.num_classes, args['num_layers'],
            args['dropout']).to(device)
evaluator = Evaluator(name='ogbn-arxiv')

In [ ]:
# Please do not change these args
# Training should take <10min using GPU runtime
# reset the parameters to initial random value
import copy

model.reset_parameters()

optimizer = torch.optim.Adam(model.parameters(), lr=args['lr'])
loss_fn = F.nll_loss

best_model = None
best_valid_acc = 0

for epoch in range(1, 1 + args["epochs"]):
  loss = train(model, data, train_idx, optimizer, loss_fn)
  result = test(model, data, split_idx, evaluator)
  train_acc, valid_acc, test_acc = result
  if valid_acc > best_valid_acc:
      best_valid_acc = valid_acc
      best_model = copy.deepcopy(model)
  print(f'Epoch: {epoch:02d}, '
        f'Loss: {loss:.4f}, '
        f'Train: {100 * train_acc:.2f}%, '
        f'Valid: {100 * valid_acc:.2f}% '
        f'Test: {100 * test_acc:.2f}%')

## Question 1: What are your `best_model` validation and test accuracies? (25 points)

Run the cell below to see the results of your best of model and save your model's predictions to a file named *ogbn-arxiv_node.csv*. You can view this file by clicking on the *Folder* icon on the left side pannel.

**Note**: Make sure you have implemented the above **GCN model** before running this test code.

In [ ]:
best_result = test(best_model, data, split_idx, evaluator, save_model_results=True)
train_acc, valid_acc, test_acc = best_result
print(f'Best model: '
      f'Train: {100 * train_acc:.2f}%, '
      f'Valid: {100 * valid_acc:.2f}% '
      f'Test: {100 * test_acc:.2f}%')

# 3 GNN: Graph Property Prediction

In this section we will create a graph neural network for graph property prediction (graph classification).

**Graph Property Prediction** is a key task in graph machine learning, where the goal is to predict properties or labels associated with **entire graphs** rather than individual nodes or edges. This task is common in applications such as molecular property prediction, social network analysis, and program verification.


- Let $\mathcal{G} = \{G_1, G_2, \dots, G_N\}$ represent a collection of graphs, where each graph $G_i = (V_i, E_i, X_i)$ consists of:
  - $V_i$ : the set of nodes,
  - $E_i$ : the set of edges,
  - $X_i \in \mathbb{R}^{|V_i| \times d}$ : the node feature matrix, where each node has $d$-dimensional features.

- The objective is to learn a function $f_\theta: \mathcal{G} \rightarrow \mathcal{Y}$ that maps each graph $G_i$ to a graph-level label $y_i \in \mathcal{Y}$.

#### **Common Applications:**
- **Molecular Property Prediction:** Predicting chemical properties or biological activities of molecules represented as graphs.
- **Social Network Analysis:** Classifying entire networks based on structure or user interactions.
- **Program Verification:** Determining properties of code represented as abstract syntax trees.

Now let us implement to address one graph classification task!



## Load and preprocess the dataset

In [ ]:
from ogb.graphproppred import PygGraphPropPredDataset, Evaluator
from torch_geometric.data import DataLoader
from tqdm.notebook import tqdm

# Load the dataset
dataset = PygGraphPropPredDataset(name='ogbg-molhiv')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device: {}'.format(device))

split_idx = dataset.get_idx_split()

# Check task type
print(f'Task type: {dataset.task_type}')

In [ ]:
# Load the dataset splits into corresponding dataloaders
# We will train the graph classification task on a batch of 32 graphs
# Shuffle the order of graphs for training set
train_loader = DataLoader(dataset[split_idx["train"]], batch_size=32, shuffle=True, num_workers=0)
valid_loader = DataLoader(dataset[split_idx["valid"]], batch_size=32, shuffle=False, num_workers=0)
test_loader = DataLoader(dataset[split_idx["test"]], batch_size=32, shuffle=False, num_workers=0)

In [ ]:
# Please DO NOT change the args
args = {
    'device': device,
    'num_layers': 5,
    'hidden_dim': 256,
    'dropout': 0.5,
    'lr': 0.001,
    'epochs': 30,
}
args

## Graph Prediction Model

### Graph Mini-Batching
Before diving into the actual model, we introduce the concept of mini-batching with graphs. In order to parallelize the processing of a mini-batch of graphs, PyG combines the graphs into a single disconnected graph data object (*torch_geometric.data.Batch*). **[torch_geometric.data.Batch](https://pytorch-geometric.readthedocs.io/en/latest/generated/torch_geometric.data.Batch.html#torch_geometric.data.Batch)** inherits from *torch_geometric.data.Data* (introduced earlier) and contains an additional attribute called `batch`.

The `batch` attribute is a vector mapping each node to the index of its corresponding graph within the mini-batch:

    batch = [0, ..., 0, 1, ..., n - 2, n - 1, ..., n - 1]

This attribute is crucial for associating which graph each node belongs to and can be used to e.g. average the node embeddings for each graph individually to compute graph level embeddings.



### Implemention

We will reuse the existing GCN model to generate `node_embeddings` and then use  `Global Pooling` over the nodes to create graph level embeddings that can be used to predict properties for the each graph. Remeber that the `batch` attribute will be essential for performining Global Pooling over our mini-batch of graphs. For more information please refer to the [Pooling Layer documentation](https://pytorch-geometric.readthedocs.io/en/latest/modules/nn.html#global-pooling-layers):

In **node property prediction**, the model focuses on learning individual node representations and predicting properties for each node. However, in **graph property prediction**, the challenge lies in **aggregating node-level information** into a **single graph-level representation** that captures the global structure and features of the entire graph.

To achieve this, we use **global pooling techniques** such as:

- **[Global Mean Pooling](https://pytorch-geometric.readthedocs.io/en/latest/generated/torch_geometric.nn.pool.global_mean_pool.html#torch_geometric.nn.pool.global_mean_pool):** Takes the average of all node embeddings in the graph.
- **[Global Max Pooling](https://pytorch-geometric.readthedocs.io/en/latest/generated/torch_geometric.nn.pool.global_max_pool.html#torch_geometric.nn.pool.global_max_pool):** Selects the maximum value across all node embeddings.
- **[Global Add Pooling](https://pytorch-geometric.readthedocs.io/en/latest/generated/torch_geometric.nn.pool.global_add_pool.html#torch_geometric.nn.pool.global_add_pool):** Sums all node embeddings to form a graph representation.

These pooling operations ensure that the model can **summarize the entire graph**, making it suitable for tasks where the **overall structure** or **interactions** between nodes are critical for prediction (e.g., predicting a molecule's solubility based on the entire molecular structure).

In this experiment, we use [Global Mean Pooling](https://pytorch-geometric.readthedocs.io/en/latest/generated/torch_geometric.nn.pool.global_mean_pool.html#torch_geometric.nn.pool.global_mean_pool). Now, we have all of the tools to implement a GCN Graph Prediction model!  


In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
import torch_geometric.transforms as T

from ogb.graphproppred.mol_encoder import AtomEncoder
from torch_geometric.nn import global_add_pool, global_mean_pool

### GCN to predict graph property
class GCN_Graph(torch.nn.Module):
    def __init__(self, hidden_dim, output_dim, num_layers, dropout):
        super(GCN_Graph, self).__init__()

        # Load encoders for Atoms in molecule graphs
        self.node_encoder = AtomEncoder(hidden_dim)

        # Node embedding model we have implemented
        # Note that the input_dim and output_dim are set to hidden_dim
        self.gnn_node = GCN(hidden_dim, hidden_dim,
            hidden_dim, num_layers, dropout, return_embeds=True)

        self.pool = None

        ############# Your code here ############

        ## Note: Initialize self.pool as a global mean pooling layer

        #########################################

        # Output layer
        self.linear = torch.nn.Linear(hidden_dim, output_dim)


    def reset_parameters(self):
      self.gnn_node.reset_parameters()
      self.linear.reset_parameters()

    def forward(self, batched_data):
        # TODO: Implement a function that takes as input a
        # mini-batch of graphs (torch_geometric.data.Batch) and
        # returns the predicted graph property for each graph.
        #
        # NOTE: Since we are predicting graph level properties,
        # your output will be a tensor with dimension equaling
        # the number of graphs in the mini-batch


        # Extract important attributes of our mini-batch
        x, edge_index, batch = batched_data.x, batched_data.edge_index, batched_data.batch
        embed = self.node_encoder(x)

        out = None

        ############# Your code here ############
        ## Note:
        ## 1. Construct node embeddings using existing GCN model
        ## 2. Use the global pooling layer to aggregate features for each individual graph
        ## 3. Use a linear layer to predict each graph's property
        ## (~3 lines of code)
        #########################################

        return out

In [ ]:
def train(model, device, data_loader, optimizer, loss_fn):
    # TODO: Implement a function that trains your model by
    # using the given optimizer and loss_fn.
    model.train()
    loss = 0

    for step, batch in enumerate(tqdm(data_loader, desc="Iteration")):
      batch = batch.to(device)

      if batch.x.shape[0] == 1 or batch.batch[-1] == 0:
          pass
      else:
        ## ignore nan targets (unlabeled) when computing training loss.
        is_labeled = batch.y == batch.y

        ############# Your code here ############
        ## Note:
        ## 1. Zero grad the optimizer
        ## 2. Feed the data into the model
        ## 3. Use `is_labeled` mask to filter output and labels
        ## 4. You may need to change the type of label to torch.float32
        ## 5. Feed the output and label to the loss_fn
        ## (~3 lines of code)
        #########################################

        loss.backward()
        optimizer.step()

    return loss.item()

In [ ]:
# The evaluation function
def eval(model, device, loader, evaluator, save_model_results=False, save_file=None):
    model.eval()
    y_true = []
    y_pred = []

    for step, batch in enumerate(tqdm(loader, desc="Iteration")):
        batch = batch.to(device)

        if batch.x.shape[0] == 1:
            pass
        else:
            with torch.no_grad():
                pred = model(batch)

            y_true.append(batch.y.view(pred.shape).detach().cpu())
            y_pred.append(pred.detach().cpu())

    y_true = torch.cat(y_true, dim = 0).numpy()
    y_pred = torch.cat(y_pred, dim = 0).numpy()

    input_dict = {"y_true": y_true, "y_pred": y_pred}

    if save_model_results:
        print ("Saving Model Predictions")

        # Create a pandas dataframe with a two columns
        # y_pred | y_true
        data = {}
        data['y_pred'] = y_pred.reshape(-1)
        data['y_true'] = y_true.reshape(-1)

        df = pd.DataFrame(data=data)
        # Save to csv
        df.to_csv('ogbg-molhiv_graph_' + save_file + '.csv', sep=',', index=False)

    return evaluator.eval(input_dict)

In [ ]:
model = GCN_Graph(args['hidden_dim'],
            dataset.num_tasks, args['num_layers'],
            args['dropout']).to(device)
evaluator = Evaluator(name='ogbg-molhiv')

In [ ]:
# Please do not change these args
# Training should take <10min using GPU runtime
import copy

model.reset_parameters()

optimizer = torch.optim.Adam(model.parameters(), lr=args['lr'])
loss_fn = torch.nn.BCEWithLogitsLoss()

best_model = None
best_valid_acc = 0

for epoch in range(1, 1 + args["epochs"]):
  print('Training...')
  loss = train(model, device, train_loader, optimizer, loss_fn)

  print('Evaluating...')
  train_result = eval(model, device, train_loader, evaluator)
  val_result = eval(model, device, valid_loader, evaluator)
  test_result = eval(model, device, test_loader, evaluator)

  train_acc, valid_acc, test_acc = train_result[dataset.eval_metric], val_result[dataset.eval_metric], test_result[dataset.eval_metric]
  if valid_acc > best_valid_acc:
      best_valid_acc = valid_acc
      best_model = copy.deepcopy(model)
  print(f'Epoch: {epoch:02d}, '
        f'Loss: {loss:.4f}, '
        f'Train: {100 * train_acc:.2f}%, '
        f'Valid: {100 * valid_acc:.2f}% '
        f'Test: {100 * test_acc:.2f}%')

## Question 2: What are your `best_model` validation and test ROC-AUC scores? (25 points)

Run the cell below to see the results of your best of model and save your model's predictions in files named *ogbg-molhiv_graph_[valid,test].csv*. Again, you can view the files by clicking on the *Folder* icon on the left side pannel.


**Note**: Make sure you have updated the above **GCN model** before running this test code.

In [ ]:
train_auroc = eval(best_model, device, train_loader, evaluator)[dataset.eval_metric]
valid_auroc = eval(best_model, device, valid_loader, evaluator, save_model_results=True, save_file="valid")[dataset.eval_metric]
test_auroc  = eval(best_model, device, test_loader, evaluator, save_model_results=True, save_file="test")[dataset.eval_metric]

print(f'Best model: '
    f'Train: {100 * train_auroc:.2f}%, '
    f'Valid: {100 * valid_auroc:.2f}% '
    f'Test: {100 * test_auroc:.2f}%')

# Submission

When you submit your assignment, you will have to download this file as an `.ipynb` file.